In [1]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
import psycopg2
#import torch
#from transformers import AutoModel, AutoTokenizer
from llama_index.core import Settings
import re

/Users/jeongwon/Desktop/Chatbot/chatbot/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#model = SentenceTransformer("jhgan/ko-sroberta-multitask", device="mps")
'''
model_name = "jhgan/ko-sroberta-multitask"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# MPS(Metal Performance Shaders) 디바이스 설정
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
model = model.to(device)
'''

Settings.embed_model = HuggingFaceEmbedding(
    model_name="jhgan/ko-sroberta-multitask"
)
#embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
#embed_model = HuggingFace
#intfloat/multilingual-e5-large: Semantic Chunking이 41시간 되고 안끝남.
#intfloat/multilingual-e5-small: 2시간 지나도 안끝남

In [4]:
try:
    table_name = 'each_paragraph'
    conn = psycopg2.connect(host='localhost', dbname='law', user='user1', password='1234', port=5432)
    cursor = conn.cursor()
    query = "SELECT name, part_num, part_name, chap_num, char_name, sec_num, sec_name, para_num, para_name, art_num, art_name, content FROM {table} order by id".format(table=table_name)
    cursor.execute(query)
    rows = cursor.fetchall()
    i = 1

    for row in rows:
        filtered_tuple = tuple(element for element in row if element is not None) #None값을 필터링하는 코드
        filtered_text = ' '.join(filtered_tuple)
        filtered_text = re.sub('(name:|part_num:|part_name:|chap_num:|char_name:|sec_num:|sec_name:|para_num:|para_name:|art_num:|art_name:|content:|None,|\n)','',filtered_text)
        embeddings = Settings.embed_model.get_text_embedding(filtered_text)
        try:
            cursor.execute("UPDATE {table} SET embedding = '{embedding}' WHERE id = {id}".format(table=table_name, embedding=str(embeddings),id=i))
        except Exception as e:
            print("Update Error: ", e)
        i+=1    
        conn.commit()

except Exception as e:
    print(f"데이터베이스 연결에 실패했습니다: {e}")
finally:
    # 데이터베이스 연결 종료
    if conn:
        cursor.close()
        conn.close()
        print("PostgreSQL 연결이 종료되었습니다.")


PostgreSQL 연결이 종료되었습니다.
